Libraries

In [30]:
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported!')

Libraries imported!


Setting up Path

In [31]:
base_path = r'C:\Users\nbush\OneDrive - University of Huddersfield\Desktop\FYP\data_image\OriginalDataset'

classes = ['MildDemented', 'ModerateDemented',
           'NonDemented', 'VeryMildDemented']

class_labels = {
    'MildDemented'    : 0,
    'ModerateDemented': 1,
    'NonDemented'     : 2,
    'VeryMildDemented': 3
}

IMG_SIZE = (128, 128)  # ← 128×128

print(f'Setup done!')
print(f'Image size : {IMG_SIZE}')
print(f'Classes    : {classes}')

Setup done!
Image size : (128, 128)
Classes    : ['MildDemented', 'ModerateDemented', 'NonDemented', 'VeryMildDemented']


Loading Images

In [32]:
images = []
labels = []

print('Loading images...')
print('-' * 40)

for cls in classes:
    cls_path = os.path.join(base_path, cls)
    
    img_files = [f for f in os.listdir(cls_path)
                 if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    
    print(f'  {cls}: {len(img_files)} images')
    
    for img_file in img_files:
        img_path = os.path.join(cls_path, img_file)
        
        img = Image.open(img_path).convert('RGB')
        img = img.resize(IMG_SIZE)
        img_array = np.array(img, dtype=np.float32)
        
        images.append(img_array)
        labels.append(class_labels[cls])

# Lists to arrays
images = np.array(images, dtype=np.float32)
labels = np.array(labels)

print('-' * 40)
print(f'Loading complete!')
print(f'Images shape : {images.shape}')
print(f'Labels shape : {labels.shape}')
print(f'Memory usage : {images.nbytes / 1024**3:.2f} GB')

Loading images...
----------------------------------------
  MildDemented: 896 images
  ModerateDemented: 64 images
  NonDemented: 3200 images
  VeryMildDemented: 2240 images
----------------------------------------
Loading complete!
Images shape : (6400, 128, 128, 3)
Labels shape : (6400,)
Memory usage : 1.17 GB


Normalization

In [33]:
images_normalized = images / 255.0

print(f'Normalization complete!')
print(f'Before : min={images.min()}, max={images.max()}')
print(f'After  : min={images_normalized.min():.1f}, max={images_normalized.max():.1f}')

Normalization complete!
Before : min=0.0, max=255.0
After  : min=0.0, max=1.0


Class Distribution

In [34]:
print('Class Distribution:')
print('-' * 40)

unique, counts = np.unique(labels, return_counts=True)
total = len(labels)

for cls_idx, count in zip(unique, counts):
    percent = count / total * 100
    print(f'  {classes[cls_idx]}: {count} ({percent:.1f}%)')

print('-' * 40)
print(f'  Total: {total}')

Class Distribution:
----------------------------------------
  MildDemented: 896 (14.0%)
  ModerateDemented: 64 (1.0%)
  NonDemented: 3200 (50.0%)
  VeryMildDemented: 2240 (35.0%)
----------------------------------------
  Total: 6400


Train/Val/Test Split

In [35]:
# Step 1 — 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    images_normalized, labels,
    test_size=0.3,
    random_state=42,
    stratify=labels
)

# Step 2 — 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print('Split complete!')
print(f'Training   : {X_train.shape}')
print(f'Validation : {X_val.shape}')
print(f'Testing    : {X_test.shape}')

print('\nTraining class distribution:')
unique, counts = np.unique(y_train, return_counts=True)
for cls_idx, count in zip(unique, counts):
    print(f'  {classes[cls_idx]}: {count}')

Split complete!
Training   : (4480, 128, 128, 3)
Validation : (960, 128, 128, 3)
Testing    : (960, 128, 128, 3)

Training class distribution:
  MildDemented: 627
  ModerateDemented: 45
  NonDemented: 2240
  VeryMildDemented: 1568


Save NPY Files

In [36]:
save_path = r'C:\Users\nbush\OneDrive - University of Huddersfield\Desktop\FYP\preprocessed'
os.makedirs(save_path, exist_ok=True)

np.save(f'{save_path}\\X_train_images.npy', X_train)
np.save(f'{save_path}\\X_val_images.npy',   X_val)
np.save(f'{save_path}\\X_test_images.npy',  X_test)
np.save(f'{save_path}\\y_train_images.npy', y_train)
np.save(f'{save_path}\\y_val_images.npy',   y_val)
np.save(f'{save_path}\\y_test_images.npy',  y_test)

print('Files saved!')
print(f'\nLocation: {save_path}')
print('\nFiles:')
for f in os.listdir(save_path):
    if f.endswith('.npy'):
        size = os.path.getsize(
            f'{save_path}\\{f}') / 1024**2
        print(f'  {f} — {size:.1f} MB')

Files saved!

Location: C:\Users\nbush\OneDrive - University of Huddersfield\Desktop\FYP\preprocessed

Files:
  X_test_images.npy — 180.0 MB
  X_train_images.npy — 840.0 MB
  X_val_images.npy — 180.0 MB
  y_test_images.npy — 0.0 MB
  y_train_images.npy — 0.0 MB
  y_val_images.npy — 0.0 MB


Verification

In [37]:
print('Verification:')
print('-' * 40)

X_tr = np.load(f'{save_path}\\X_train_images.npy')
y_tr = np.load(f'{save_path}\\y_train_images.npy')

print(f'X_train shape  : {X_tr.shape}')
print(f'y_train shape  : {y_tr.shape}')
print(f'Image size     : {X_tr.shape[1]}×{X_tr.shape[2]}')
print(f'Channels       : {X_tr.shape[3]}')
print(f'Pixel range    : {X_tr.min():.1f} to {X_tr.max():.1f}')

# Size confirm karo
if X_tr.shape[1] == 128:
    print('\nImage size 128×128 confirmed!')
    print('Preprocessing Complete!')
    print('Next → CNN Model Training!')
else:
    print(f'\nWrong size: {X_tr.shape[1]}')
    print('Cell 2 mein IMG_SIZE = (128,128) karo!')

Verification:
----------------------------------------
X_train shape  : (4480, 128, 128, 3)
y_train shape  : (4480,)
Image size     : 128×128
Channels       : 3
Pixel range    : 0.0 to 1.0

Image size 128×128 confirmed!
Preprocessing Complete!
Next → CNN Model Training!
